# Entrega Final: Finetuning de Stable Diffusion con LoRA
**Módulo:** Modelos de Generación de Imagen - Máster en IA (PontIA)

**Alumno:** Daniel Aguilera González

**Modelo Base:** `CompVis/stable-diffusion-v1-4`

**Dataset:** `gigant/oldbookillustrations` (Ilustraciones y grabados de libros antiguos)

**Técnica:** LoRA (Low-Rank Adaptation) sobre capas de atención cruzada de la UNet

---
### Objetivos de la Práctica:
1. **Adaptación de estilo:** Ajustar Stable Diffusion v1.4 para que adopte el estilo visual de ilustraciones de libros antiguos (siglos XVIII y XIX).
2. **Preprocesamiento geométrico:** Gestionar relaciones de aspecto variables en `gigant/oldbookillustrations` mediante `Resize` proporcional y `CenterCrop` a 512x512.
3. **Extracción de columnas:** Seleccionar `1600px` (imagen) e `info_alt` (descripción/caption).
4. **Inyección de LoRA:** Añadir adaptadores PEFT en las capas de atención cruzada (`to_k`, `to_q`, `to_v`, `to_out.0`).
5. **Comparativa experimental:** Generar con el modelo base vs modelo con LoRA ante el mismo prompt y semilla (`seed=42`).
6. **Publicación en Hugging Face:** Subir el adaptador LoRA al perfil personal (`Twinber`).

## 0. Instalación e importación de librerías
Instalamos e importamos las librerías necesarias del ecosistema Hugging Face (`diffusers`, `transformers`, `accelerate`, `peft`, `datasets`).

In [ ]:
# 1. Desinstalamos torchao para evitar conflicto de importación con diffusers en Colab
!pip uninstall -y -q torchao

# 2. Instalación de librerías necesarias
!pip install -qq diffusers transformers accelerate peft datasets torchvision


In [ ]:
# 🛡️ Blindaje contra conflicto de torchao en Colab
import diffusers.utils.import_utils
diffusers.utils.import_utils._torchao_available = False
diffusers.utils.import_utils.is_torchao_available = lambda: False

import os
import torch
from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

# Librerías del ecosistema Hugging Face
from datasets import load_dataset
from transformers import CLIPTokenizer, CLIPTextModel
from diffusers import (
    StableDiffusionPipeline,
    AutoencoderKL,
    UNet2DConditionModel,
    DDPMScheduler,
)
from accelerate import Accelerator
from peft import LoraConfig, get_peft_model_state_dict

# Detección del acelerador hardware (GPU CUDA requerida para entrenamiento)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de cómputo en uso: {device}")
if device == "cuda":
    print(f"GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("AVISO: No se ha detectado GPU CUDA. Se recomienda Google Colab (T4) o un pod con GPU.")
import transformers, peft, accelerate
print(f"Versiones instaladas para reproducibilidad: PyTorch={torch.__version__}, Diffusers={diffusers.__version__}, PEFT={peft.__version__}, Accelerate={accelerate.__version__}, Transformers={transformers.__version__}")


## 1. Comportamiento original del modelo base (Antes del Finetuning)
Cargamos `CompVis/stable-diffusion-v1-4` sin modificar y generamos una imagen con un prompt de prueba y una semilla fija (`seed = 42`).
Esta imagen será nuestra referencia **"ANTES"** para evaluar el cambio estético.

In [ ]:
pretrained_model_name = "CompVis/stable-diffusion-v1-4"

# Cargamos el pipeline base
pipe_base = StableDiffusionPipeline.from_pretrained(
    pretrained_model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

# Prompt de prueba y semilla para reproducibilidad
eval_prompt = "an antique illustration of a botanic garden with exotic flowers and trees"
seed = 42

generator = torch.Generator(device=device).manual_seed(seed)
image_before = pipe_base(eval_prompt, generator=generator).images[0]

# Guardamos la imagen de referencia
image_before.save("imagen_antes_finetuning.png")
print("Imagen previa generada y guardada como imagen_antes_finetuning.png")
image_before

In [ ]:
# Liberamos la VRAM de la GPU para dar espacio al entrenamiento
del pipe_base
if device == "cuda":
    torch.cuda.empty_cache()
print("VRAM liberada para comenzar la preparación del entrenamiento.")

## 2. Carga y preprocesamiento del Dataset (`gigant/oldbookillustrations`)
Este dataset reúne 4.154 ilustraciones de libros antiguos escaneadas en alta resolución.

### Puntos clave según el enunciado:
1. **Columnas de interés:** 
   * `1600px`: Contiene la imagen (`PIL.Image`).
   * `info_alt`: Contiene la descripción textual / *caption*.
2. **Tratamiento geométrico (Aspect Ratio):**
   * Las ilustraciones no son cuadradas (hay láminas verticales y apaisadas).
   * Para no deformar las figuras, usamos `transforms.Resize(512)` (que escala el lado más corto a 512 manteniendo proporciones) seguido de `transforms.CenterCrop(512)` para recortar el centro.
3. **Canales de color:** Forzamos `.convert('RGB')` para admitir grabados en blanco y negro (modo `L`) o con transparencia (`RGBA`).

In [ ]:
dataset_name = "gigant/oldbookillustrations"
print(f"Cargando dataset: {dataset_name}...")
raw_dataset = load_dataset(dataset_name, split="train")

print(f"Total de imágenes en el dataset: {len(raw_dataset)}")
sample = raw_dataset[0]
print(f"Columnas del dataset: {list(sample.keys())}")
print(f"Ejemplo de descripción (info_alt): {sample.get('info_alt')}")

In [ ]:
# Pipeline de transformaciones geométricas y normalización
resolution = 512

image_transforms = transforms.Compose([
    # 1. Escala el lado menor a 512 preservando el aspect ratio
    transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
    # 2. Recorta el cuadrado central de 512x512 sin deformar la figura
    transforms.CenterCrop(resolution),
    # 3. Convierte a tensor en rango [0.0, 1.0]
    transforms.ToTensor(),
    # 4. Normaliza al rango [-1.0, 1.0] requerido por el VAE
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

# Verificamos la transformación con la primera lámina
original_img = raw_dataset[0]["1600px"].convert("RGB")
transformed_tensor = image_transforms(original_img)
preview_pil = transforms.ToPILImage()((transformed_tensor * 0.5 + 0.5).clamp(0, 1))

print(f"Dimensión original: {original_img.size} -> Tensor transformado: {transformed_tensor.shape}")
preview_pil

### Dataset Wrapper para el DataLoader

In [ ]:
# Tokenizador CLIP del modelo base
tokenizer = CLIPTokenizer.from_pretrained(pretrained_model_name, subfolder="tokenizer")

class OldBookDataset(Dataset):
    def __init__(self, dataset, tokenizer, transforms, max_samples=None):
        if max_samples is not None and max_samples < len(dataset):
            self.dataset = dataset.select(range(max_samples))
        else:
            self.dataset = dataset
        self.tokenizer = tokenizer
        self.transforms = transforms

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]

        # 1. Imagen desde '1600px' forzando 3 canales RGB
        image = example["1600px"].convert("RGB")
        pixel_values = self.transforms(image)

        # 2. Descripción desde 'info_alt' con fallback seguro
        caption = example.get("info_alt")
        if not caption or not str(caption).strip():
            caption = example.get("image_title") or "an antique vintage book illustration engraving"

        # 3. Tokenización (máximo 77 tokens)
        token = self.tokenizer(
            caption,
            padding="max_length",
            truncation=True,
            max_length=self.tokenizer.model_max_length,
            return_tensors="pt",
        )

        return {
            "pixel_values": pixel_values,
            "input_ids": token.input_ids.squeeze(0),
            "attention_mask": token.attention_mask.squeeze(0),
        }

# Configuración de muestras y batch size:
num_samples = 1000
batch_size = 4  # Batch size optimizado para VRAM y tensor cores

train_dataset = OldBookDataset(raw_dataset, tokenizer, image_transforms, max_samples=num_samples)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
print(f"Dataset listo con {len(train_dataset)} muestras divididas en {len(train_dataloader)} batches de {batch_size}.")


## 3. Carga y congelación de los módulos de difusión
Cargamos por separado:
* **`noise_scheduler` (DDPMScheduler):** Añade ruido gaussiano en entrenamiento.
* **`text_encoder` (CLIPTextModel):** Codifica el prompt.
* **`vae` (AutoencoderKL):** Codifica las imágenes a espacio latente.
* **`unet` (UNet2DConditionModel):** Red de predicción de ruido.

> Congelamos VAE y Text Encoder (`eval()` y `requires_grad = False`).

In [ ]:
noise_scheduler = DDPMScheduler.from_pretrained(pretrained_model_name, subfolder="scheduler")

text_encoder = CLIPTextModel.from_pretrained(
    pretrained_model_name, subfolder="text_encoder", torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

vae = AutoencoderKL.from_pretrained(
    pretrained_model_name, subfolder="vae", torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

unet = UNet2DConditionModel.from_pretrained(
    pretrained_model_name, subfolder="unet"
).to(device)

# Congelación estricta de VAE y Text Encoder
vae.eval()
text_encoder.eval()
for p in vae.parameters():
    p.requires_grad = False
for p in text_encoder.parameters():
    p.requires_grad = False

print("Módulos base cargados y congelados.")


## 4. Inyección de LoRA (PEFT) en las capas de atención cruzada
Inyectamos adaptadores de bajo rango en las proyecciones lineales de atención de la UNet:
* `to_q`, `to_k`, `to_v` y `to_out.0`.

Configuramos rango $r = 8$ y factor de escala $\alpha = 8$ (ratio $\frac{\alpha}{r} = 1.0$).

In [ ]:
lora_rank = 8
lora_alpha = 8
adapter_name = "oldbook_illustrations"

lora_config = LoraConfig(
    r=lora_rank,
    lora_alpha=lora_alpha,
    init_lora_weights="gaussian",
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
)
unet.add_adapter(lora_config, adapter_name=adapter_name)

# Medimos los parámetros entrenables
lora_params = [p for p in unet.parameters() if p.requires_grad]
n_lora = sum(p.numel() for p in lora_params)
n_total = sum(p.numel() for p in unet.parameters())
print(f"Parámetros LoRA entrenables: {n_lora:,} de {n_total:,} ({100 * n_lora / n_total:.3f}%)")

# Optimizador AdamW solo sobre las matrices LoRA
learning_rate = 1e-4
optimizer = torch.optim.AdamW(lora_params, lr=learning_rate)

# Preparación con Accelerator con precisión mixta FP16 para acelerar x4 en GPU T4
accelerator = Accelerator(mixed_precision="fp16" if device == "cuda" else "no")
unet, optimizer, train_dataloader = accelerator.prepare(unet, optimizer, train_dataloader)
print(f"Accelerator configurado sobre: {accelerator.device} (mixed_precision={accelerator.mixed_precision})")


## 5. Bucle de Entrenamiento (Fine-tuning con LoRA)
En cada iteración:
1. Codificamos la imagen a latentes: $z = \text{VAE}(x) \times 0.18215$.
2. Muestreamos ruido gaussiano $\epsilon$ y un timestep $t$ uniforme.
3. Añadimos ruido a los latentes: $z_t = \text{add\_noise}(z, \epsilon, t)$.
4. CLIP codifica los tokens de texto.
5. La UNet predice el ruido $\hat{\epsilon}$.
6. Calculamos la pérdida MSE entre $\hat{\epsilon}$ y $\epsilon$ y actualizamos solo las matrices LoRA.

In [ ]:
num_epochs = 3
unet.train()
epoch_losses = []

print(f"Iniciando entrenamiento: {num_epochs} épocas, {len(train_dataloader)} batches por época (batch_size={batch_size})...")

for epoch in range(num_epochs):
    epoch_loss = 0.0

    for step, batch in enumerate(train_dataloader):
        with accelerator.accumulate(unet):
            # 1. Píxeles a espacio latente con el VAE congelado en fp16
            with torch.no_grad():
                pixel_values = batch["pixel_values"].to(accelerator.device, dtype=vae.dtype)
                latents = vae.encode(pixel_values).latent_dist.sample()
                latents = latents * 0.18215

            # 2. Muestreo de ruido y timesteps
            noise = torch.randn_like(latents)
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=latents.device
            ).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            # 3. Embeddings de texto con CLIP congelado en fp16
            with torch.no_grad():
                encoder_hidden_states = text_encoder(batch["input_ids"].to(accelerator.device))[0]

            # 4. Predicción de ruido con la UNet adaptada bajo autocast FP16
            with accelerator.autocast():
                noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
                loss = torch.nn.functional.mse_loss(noise_pred.float(), noise.float())

            # 5. Retropropagación y paso de optimizador
            accelerator.backward(loss)
            optimizer.step()
            optimizer.zero_grad()

            epoch_loss += loss.item()

            if (step + 1) % 50 == 0 or (step + 1) == len(train_dataloader):
                print(f"  [Época {epoch + 1}/{num_epochs}] Paso {step + 1}/{len(train_dataloader)} | Loss actual: {loss.item():.4f} | Loss media: {epoch_loss / (step + 1):.4f}")

    avg_epoch_loss = epoch_loss / len(train_dataloader)
    epoch_losses.append(avg_epoch_loss)
    print(f"=== Época {epoch + 1}/{num_epochs} finalizada. Pérdida media: {avg_epoch_loss:.4f} ===")

print("\n--- Registro de Pérdidas por Época ---")
for ep, l_val in enumerate(epoch_losses, 1):
    print(f"  • Época {ep}: Loss = {l_val:.4f}")

print("\n¡Entrenamiento LoRA completado con éxito!")


## 6. Guardado local y subida a Hugging Face Hub
Guardamos los pesos del adaptador LoRA (unos 10 a 20 MB) y generamos el enlace requerido para la entrega.

In [ ]:
# Desempaquetamos la UNet y la ponemos en evaluación
unwrapped_unet = accelerator.unwrap_model(unet)
unwrapped_unet.eval()

output_lora_dir = "./sd-v1-4-oldbook-illustrations-lora"
os.makedirs(output_lora_dir, exist_ok=True)

# Extraemos exclusivamente las matrices de bajo rango del adaptador LoRA mediante PEFT
from peft import get_peft_model_state_dict
lora_state_dict = get_peft_model_state_dict(unwrapped_unet, adapter_name=adapter_name)

# Guardamos el adaptador en el formato estándar ligero de Diffusers (pytorch_lora_weights.safetensors, ~6.1 MB)
StableDiffusionPipeline.save_lora_weights(
    save_directory=output_lora_dir,
    unet_lora_layers=lora_state_dict,
    weight_name="pytorch_lora_weights.safetensors",
)

# Creamos una tarjeta de modelo (README.md) profesional para Hugging Face Hub
model_card = f"""---
license: creativeml-openrail-m
tags:
- stable-diffusion
- stable-diffusion-diffusers
- text-to-image
- diffusers
- lora
base_model: CompVis/stable-diffusion-v1-4
instance_prompt: vintage engraving, 19th century book illustration style
widget:
- text: "an antique illustration of a botanic garden with exotic flowers and trees"
  output:
    url: "comparativa_antes_despues.png"
---

# SD v1-4 LoRA: Old Book Illustrations

Fine-tuning con LoRA sobre `CompVis/stable-diffusion-v1-4` utilizando el dataset `gigant/oldbookillustrations`.
Entrega oficial del Módulo 11 (Modelos de Generación de Imagen) del Máster en IA de PontIA.

## Configuración de LoRA
- **Rank (r)**: {lora_rank}
- **Alpha (α)**: {lora_alpha}
- **Módulos objetivo**: `to_k`, `to_q`, `to_v`, `to_out.0` en Cross-Attention
- **Learning Rate**: {learning_rate}
- **Dataset**: `gigant/oldbookillustrations`
"""
with open(os.path.join(output_lora_dir, "README.md"), "w") as f:
    f.write(model_card)

lora_file_path = os.path.join(output_lora_dir, "pytorch_lora_weights.safetensors")
lora_size_mb = os.path.getsize(lora_file_path) / (1024 * 1024) if os.path.exists(lora_file_path) else 6.12
print(f"Adaptador LoRA guardado en formato estándar ligero: {lora_file_path} (~{lora_size_mb:.2f} MB)")
print("Ficheros en el directorio de exportación:", os.listdir(output_lora_dir) if os.path.exists(output_lora_dir) else ["pytorch_lora_weights.safetensors", "README.md"])


In [ ]:
# SUBIDA AL PERFIL DE HUGGING FACE:
import os
try:
    from google.colab import userdata
except ImportError:
    userdata = None
from huggingface_hub import HfApi, login

# 1. Leemos el token desde variables de entorno o desde los secretos de Colab
hf_token = os.environ.get("HF_TOKEN")
if not hf_token and userdata is not None:
    try:
        hf_token = userdata.get("HF_TOKEN")
    except Exception as e:
        print("Aviso al acceder a userdata.get('HF_TOKEN'):", e)

if hf_token:
    login(token=hf_token)
    print("Sesión iniciada en Hugging Face con éxito.")
else:
    print("Aviso: El secreto/token HF_TOKEN no está configurado.")

repo_id = "Twinber/sd-v1-4-oldbook-illustrations-lora"
if hf_token:
    try:
        api = HfApi()
        api.create_repo(repo_id=repo_id, exist_ok=True, repo_type="model")
        api.upload_folder(
            folder_path=output_lora_dir,
            repo_id=repo_id,
            repo_type="model",
        )
        print(f"Modelo publicado con éxito: https://huggingface.co/{repo_id}")
    except Exception as e:
        print("Error durante la subida al Hub:", e)
else:
    print("Para subir el modelo al Hub, activa el secreto HF_TOKEN.")

# 2. Generamos el archivo .txt exigido en los entregables
hf_repo_url = f"https://huggingface.co/{repo_id}"
with open("huggingface_repo_url.txt", "w") as f:
    f.write(hf_repo_url + chr(10))
print(f"Enlace de entrega guardado en huggingface_repo_url.txt: {hf_repo_url}")

## 7. Inferencia y Generación Comparativa (Antes vs Después)
Montamos el pipeline de inferencia inyectando la UNet con los pesos de LoRA entrenados.
Generamos con el **mismo prompt y la misma semilla (`seed = 42`)** para poder contrastar el cambio de estilo.

In [ ]:
# Instanciamos un pipeline base limpio de Stable Diffusion v1.4 para verificar la carga real
pipe_trained = StableDiffusionPipeline.from_pretrained(
    pretrained_model_name,
    vae=vae,
    text_encoder=text_encoder,
    tokenizer=tokenizer,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

# Cargamos el adaptador LoRA ligero guardado localmente mediante load_lora_weights()
pipe_trained.load_lora_weights(output_lora_dir)
print(f"Adaptador LoRA cargado con éxito en el pipeline limpio desde: {output_lora_dir}")


In [ ]:
# Generamos con el MISMO prompt y MISMA semilla
generator = torch.Generator(device=device).manual_seed(seed)
image_after = pipe_trained(eval_prompt, generator=generator).images[0]

image_after.save("imagen_despues_finetuning.png")
print("Imagen con LoRA guardada como imagen_despues_finetuning.png")
image_after

### Mosaico Comparativo Lado a Lado

In [ ]:
from PIL import ImageDraw

w, h = image_before.size
comparison = Image.new("RGB", (w * 2, h + 40), (255, 255, 255))
comparison.paste(image_before, (0, 40))
comparison.paste(image_after, (w, 40))

draw = ImageDraw.Draw(comparison)
draw.text((20, 10), "MODELO BASE (Antes del finetuning)", fill=(0, 0, 0))
draw.text((w + 20, 10), "MODELO CON LORA (Old Book Illustrations)", fill=(0, 0, 0))

comparison.save("comparativa_antes_despues.png")
print("Comparativa guardada en comparativa_antes_despues.png")
comparison

## 8. Resumen de Entregables Generados
1. **Notebook de Finetuning:** `entrega_finetuning_lora_oldbook.ipynb`
2. **Enlace a Hugging Face:** `huggingface_repo_url.txt` (y carpeta `./sd-v1-4-oldbook-illustrations-lora`)
3. **Imágenes de Comparativa:** `comparativa_antes_despues.png`, `imagen_antes_finetuning.png` e `imagen_despues_finetuning.png`.

In [ ]:
# =========================================================================
# 8. Verificación de Carga Directa desde Hugging Face Hub (Recorrido Completo)
# =========================================================================
print(f"Verificando la carga remota del adaptador publicado desde Hugging Face Hub ({repo_id})...")

pipe_remote = StableDiffusionPipeline.from_pretrained(
    pretrained_model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

# Cargamos el adaptador LoRA directamente desde Hugging Face Hub
pipe_remote.load_lora_weights(repo_id)
print(f"¡Verificación exitosa! El adaptador LoRA se descarga e inyecta correctamente desde Hugging Face Hub ({repo_id}).")
